# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RamaKousalya/FlyRank/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

files = list(Path(".").rglob("*.csv"))
df = pd.read_csv(files[0], header=None)

X = df.iloc[:, 1:]
y = df.iloc[:, 0]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=.2, random_state=42, stratify=y
)

model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)
pred = model.predict(X_test)

print("Normal split accuracy:", accuracy_score(y_test, pred))
print("Normal split F1:", f1_score(y_test, pred, average="weighted"))


Normal split accuracy: 0.881
Normal split F1: 0.8805657111797588


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Honest holdout: no shuffling, preserving the original row order
cut = int(len(df) * .8)

X_train_honest = X.iloc[:cut]
X_test_honest = X.iloc[cut:]
y_train_honest = y.iloc[:cut]
y_test_honest = y.iloc[cut:]

honest_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

honest_model.fit(X_train_honest, y_train_honest)
honest_pred = honest_model.predict(X_test_honest)

comparison = pd.DataFrame({
    "split": ["random", "honest_ordered"],
    "accuracy": [
        accuracy_score(y_test, pred),
        accuracy_score(y_test_honest, honest_pred)
    ],
    "f1": [
        f1_score(y_test, pred, average="weighted"),
        f1_score(y_test_honest, honest_pred, average="weighted")
    ]
})

display(comparison)


,split,accuracy,f1
0,random,0.881,0.880566
1,honest_ordered,0.889,0.888601


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("LEAKAGE AUDIT")
print("Target column excluded from X:", 0 not in X.columns)
print("Duplicate rows:", df.duplicated().sum())
print("Missing values:", df.isna().sum().sum())

errors = X_test_honest.copy()
errors["actual"] = y_test_honest.values
errors["predicted"] = honest_pred
errors["wrong"] = errors["actual"] != errors["predicted"]

print("\nFailure examples:")
display(errors[errors["wrong"]].head(10))


LEAKAGE AUDIT
Target column excluded from X: True
Duplicate rows: 0
Missing values: 0

Failure examples:


,1,2,3,4,5,6,7,8,9,10,...,778,779,780,781,782,783,784,actual,predicted,wrong
16017,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,2,4,True
16023,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,8,9,True
16042,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,7,4,True
16059,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,9,4,True
16063,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,5,8,True
16073,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,8,True
16076,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,8,2,True
16081,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,6,3,True
16085,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,9,4,True
16126,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,9,3,True


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("""
PAPER FINDING 1:
The paper reports a research finding based on its stated labeling process.
Methodology question: How were the labels created, and what independent evidence
supports their correctness?

PAPER FINDING 2:
The paper reports model performance using its validation design.
Methodology question: Does the validation split prevent information from related
examples appearing in both training and evaluation data?

MY CLAIM:
The Week-5 model showed measured performance on the available holdout data.
The honest ordered split provides a directional check, not proof of real-world
generalization.

LIMITATION:
This dataset does not expose a clear client or time identifier, so a true
client-grouped or time-aware validation could not be established from these
columns alone.
""")



PAPER FINDING 1:
The paper reports a research finding based on its stated labeling process.
Methodology question: How were the labels created, and what independent evidence
supports their correctness?

PAPER FINDING 2:
The paper reports model performance using its validation design.
Methodology question: Does the validation split prevent information from related
examples appearing in both training and evaluation data?

MY CLAIM:
The Week-5 model showed measured performance on the available holdout data.
The honest ordered split provides a directional check, not proof of real-world
generalization.

LIMITATION:
This dataset does not expose a clear client or time identifier, so a true
client-grouped or time-aware validation could not be established from these
columns alone.



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.